# **QT colour signal processing and viz on ECGRDVQ**



In [ ]:
!pip install neurokit2
!pip install scipy
!pip install openpyxl

In [ ]:
import xml.etree.ElementTree as ET

import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import glob
from scipy.signal import find_peaks
from scipy.signal import butter
from scipy import signal, datasets
import scipy

import matplotlib.colors as colors
import openpyxl
import plotnine
import seaborn as sns
from plotnine import *
import neurokit2 as nk

# -

import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
import zipfile
import shutil

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
import pywt

def wavelet_denoise(data, wavelet='sym8', level=5):
    # 1. Decompose the signal
    coeffs = pywt.wavedec(data, wavelet, level=level)

    # 2. Estimate noise threshold using universal threshold
    # Estimate noise from the highest frequency detail coefficients (coeffs[-1])
    noise_sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = noise_sigma * np.sqrt(2 * np.log(len(data)))

    # 3. Apply soft thresholding (keep approximation coefficients coeffs[0] untouched)
    thresholded_coeffs = [coeffs[0]]
    for detail_coeff in coeffs[1:]:
        thresholded_coeffs.append(pywt.threshold(detail_coeff, value=threshold, mode='soft'))

    # 4. Reconstruct the signal
    denoised_signal = pywt.waverec(thresholded_coeffs, wavelet)
    return denoised_signal

In [ ]:
from scipy.signal import butter, sosfiltfilt

def remove_baseline_wander(ecg_signal, fs, cutoff=0.5):
    """
    Removes baseline wander from an ECG signal using a zero-phase high-pass filter.
    :param ecg_signal: 1D array of the raw ECG signal
    :param fs: Sampling frequency in Hz
    :param cutoff: Cutoff frequency for the high-pass filter
    :return: 1D array of the filtered ECG signal
    """
    # Design a 2nd order Butterworth filter (2nd order is generally standard for ECG)
    sos = butter(N=2, Wn=cutoff, btype='highpass', fs=fs, output='sos')

    # Apply the filter forward and backward for zero phase distortion
    filtered_ecg = sosfiltfilt(sos, ecg_signal)

    return filtered_ecg

In [ ]:
def bandpass_filter(signal_data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    y = signal.filtfilt(b, a, signal_data)
    return y

def detect_r_peaks(ecg_signal, fs):
    # 1. Bandpass filter to isolate QRS frequencies (suppresses P/T waves)
    filtered_signal = bandpass_filter(ecg_signal, lowcut=5.0, highcut=15.0, fs=fs, order=2)

    # 2. Derivative: emphasizes high slopes (R-peaks), suppresses slow P/T waves
    derivative = np.diff(filtered_signal)
    derivative = np.append(derivative, 0) # align length

    # 3. Squaring: non-linear amplification of higher frequencies
    squared = derivative ** 2

    # 4. Moving Window Integration (MWI): captures QRS complex width
    integration_window = int(fs * 0.120) # 120ms window
    integrated = np.convolve(squared, np.ones(integration_window)/integration_window, mode='same')

    # 5. Peak Detection using Scipy
    # Min distance between R-peaks (e.g., 200ms is max heart rate of 300bpm)
    distance = int(fs * 0.200)
    peaks, _ = signal.find_peaks(integrated, distance=distance, prominence=0.2 * np.max(integrated))

    return peaks

In [ ]:
def refine_r_peaks(sig, peaks, fs, win_ms=50):
    """Move each detected peak to the true R apex within +/- win_ms."""
    sig = np.asarray(sig)
    w = int(round(win_ms * fs / 1000))
    return np.array([max(p - w, 0) + int(np.argmax(sig[max(p - w, 0):p + w + 1])) for p in peaks])

In [ ]:
# Folder containing your selected CSV files
input_folder = '/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/raw_signals'

# Process every CSV file
for filename in os.listdir(input_folder):

    if not filename.lower().endswith('.csv'):
        continue

    csv_path = os.path.join(input_folder, filename)
    file_name = filename

    # Read CSV
    ECG = pd.read_csv(csv_path, skiprows=1)
    ECG.columns = ["time", "mv"]
    ECG_in_mv = ECG["mv"]
    ECG_signal = ECG["mv"]

    fs = 1000 # Sampling Rate in Hz Frequency

    increment = 1000/fs # if 500HZ then 2, if 250Hz then 4

    time = np.arange(0, len(ECG_signal), 1)
    # To convert a frequency of 500 Hz (5000 rows) to milliseconds/seconds
    time_s = time/fs
    time_ms = time/fs * 1000

    ECG_df = pd.DataFrame({'ECG_mv':ECG_in_mv, 'time':time_s})

    #Fixing the Baseline
    filtered_ecg = remove_baseline_wander(ECG_signal, fs)
    cleaned_ecg = wavelet_denoise(filtered_ecg)
    cleaned_ecg = nk.ecg_clean(cleaned_ecg, sampling_rate=fs)

    lowpassed = scipy.signal.medfilt(cleaned_ecg, kernel_size=fs+1)
    highpassed = cleaned_ecg - lowpassed

    R_peaks_index = refine_r_peaks(highpassed, detect_r_peaks(highpassed, fs), fs)

    highpassed_df = pd.DataFrame(highpassed)
    cleaned_ecg = highpassed_df[0]
    R_peaks_mv = cleaned_ecg[R_peaks_index]
    plt.plot(highpassed)
    plt.plot(R_peaks_mv,'o')
    plt.show()

    # To convert a frequency of in Hz to milliseconds to calculate RR-interval
    R_peaks_ms = R_peaks_index/fs * 1000

    RR_interval = []

    for i in range(len(R_peaks_ms)-1):
      RR_interval.append(R_peaks_ms[i+1] - R_peaks_ms[i])

    # RR_interval = [x for x in RR_interval if 500 <= x <= 1600]
    RR = np.mean(RR_interval)
    HR = 60000 / RR
    print(HR)

    _, rpeaks = nk.ecg_peaks(cleaned_ecg, sampling_rate=fs, method="neurokit")
    signal_dwt, waves_dwt = nk.ecg_delineate(cleaned_ecg, rpeaks, sampling_rate=fs, method="dwt", show=False, show_type="all")


    ECG_R_Peaks = rpeaks['ECG_R_Peaks']
    print(ECG_R_Peaks)

    ECG_Q_Peaks = waves_dwt['ECG_Q_Peaks']

    if np.isnan(HR)==True:
     R_peaks_index = ECG_R_Peaks
     R_peaks_mv = cleaned_ecg[ECG_R_Peaks]
     R_peaks_ms = R_peaks_index/fs * 1000
     R_peaks_s = R_peaks_index/fs
     RR_interval = []
     for i in range(len(R_peaks_ms)-1):
         RR_interval.append(R_peaks_ms[i+1] - R_peaks_ms[i])
    else:
     R_peaks_ms = R_peaks_index/fs * 1000
     R_peaks_s = R_peaks_index/fs

    # RR_interval = [x for x in RR_interval if 500 <= x <= 1600]
    RR = np.mean(RR_interval)
    HR = 60000 / RR
    print(HR)


    QTnomogram = pd.read_csv("/content/drive/My Drive/Colab Notebooks/UKB_Research/XAI/QTnomogram.csv")

    # Access HR and QT_pred columns from QTnomogram
    HR_nomogram = QTnomogram['HR']
    QT_nomogram = QTnomogram['QT_pred']

    index_HR = np.argmin(np.abs(HR_nomogram - HR))
    hr_value = HR_nomogram[index_HR]
    QT_nomogram_value = QT_nomogram[index_HR]

    Q_wave_index = int(round(28 * fs / 1000))   # 28 ms at any sampling rate

    Rpeaks_df = pd.DataFrame({'R_peaks_index':R_peaks_index,'R_peaks_ms':R_peaks_ms, 'R_peaks_s':R_peaks_s,'Rpeaks_mv':R_peaks_mv*0.005,'RR':RR,'HR':HR, 'QT_value':QT_nomogram_value,'Q_wave_index':Q_wave_index}) # Also R-peak needs to be converted from micro uv to mv
    Rpeaks_df.to_csv("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/QT_coloured_ECG_signal_v2/Rpeaks/"+ file_name + "_.csv")

    #Create QT Colouring Vector
    UpperColourLimit = QT_nomogram_value + (40 * 2)
    LowerColourlimit = QT_nomogram_value - (40 * 6)

    maximumColourVector = UpperColourLimit

    if HR >= 85:
      maximumColourVector = QT_nomogram_value + 40
    else:
     maximumColourVector = UpperColourLimit

    # initialize_colour_vector
    length = len(ECG_in_mv)
    colourVector = [None] * length
    value = [0] * length


    for R in range(len(R_peaks_index)):
      if R_peaks_index[R] >= Q_wave_index:
        index = int(R_peaks_index[R]) - Q_wave_index
      else:
        index = int(R_peaks_index[R]) - Q_wave_index
      for i in range(0, int(maximumColourVector)):
         if ((index+i)>=0 and (index+i)<len(ECG_in_mv)): # condition to avoid list assignment index out of range error
            colourVector[index + i] = value[i]
            value[i+1] = value[i] + increment

    ##### process color vector
    newvector_length = len(ECG_in_mv)
    # Create a copy of colourVector with the desired length
    newcolourVector = colourVector[:newvector_length]

    # Set None values to np.nan
    newcolourVector = [x if x is not None else np.nan for x in newcolourVector]

    # replace values above upperlimit and below lower limit with nan
    for i in range(len(newcolourVector)):
     if (newcolourVector[i] < LowerColourlimit) or (newcolourVector[i] > UpperColourLimit):
        newcolourVector[i] =  np.nan

    ECG_with_colour_vector = pd.DataFrame({'time':time_s,'mv':cleaned_ecg,'newcolourVector':newcolourVector})
    ECG_with_colour_vector.to_csv("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/QT_coloured_ECG_signal_v2/" + file_name + "_.csv")

print("Finished!")

In [ ]:
import glob

os.chdir("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/QT_coloured_ECG_signal_v2/")
os.getcwd()

path=os.getcwd()
images=glob.glob(os.path.join(path,"*.csv"))
len(images)

# **QT-nomogram, QTcB, QTcF: **

In [ ]:
# =============================================================================
#  Colour-vector generation for THREE rate-correction rules.
#
#  Changes from the original:
#    - SEX is looked up from the mapping table, to pick 450 (M) / 460 (F)
#    - three anchors instead of one:
#         nomogram  QT_nomogram_value          (unchanged)
#         QTcB      thr * sqrt(RR_s)
#         QTcF      thr * RR_s**(1/3)
#    - the ms-since-Q ramp is built ONCE (it does not depend on the rule),
#      then clipped three ways -> three colour vectors in one output CSV
#    - the Rpeaks CSV carries all three anchors so the R renderer can read them
#
#  Output columns: newcolourVector_nom, newcolourVector_qtcb, newcolourVector_qtcf
#  (newcolourVector is kept as an alias of the nomogram one, so existing R code
#   that reads `newcolourVector` keeps working unchanged.)
# =============================================================================

import os
import numpy as np
import pandas as pd

# ---- configuration ----------------------------------------------------------
input_folder = '/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/raw_signals'
OUT_SIGNAL = '/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/QT_coloured_ECG_signal_v3/'
OUT_RPEAKS = OUT_SIGNAL + 'Rpeaks/'
os.makedirs(OUT_RPEAKS, exist_ok=True)
MAPPING_CSV  = '/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/metadata/mapping_table_with_outcome_and_distance.csv'

QTC_PROLONG_M = 450.0    # ms, men
QTC_PROLONG_F = 460.0    # ms, women

BAND        = 40.0       # ms per colour band
BANDS_BELOW = 6          # index 1 sits 6 bands below the anchor
BANDS_ABOVE = 2          # index 9 sits 2 bands above
BANDS_ABOVE_FAST = 1     # at HR >= 85 the scale is truncated, as in the original

RULES = ('nom', 'qtcb', 'qtcf')

# ---- sex lookup -------------------------------------------------------------
_map = pd.read_csv(MAPPING_CSV)
_map = _map.drop_duplicates('ECG_recording_number')
SEX_OF = dict(zip(_map['ECG_recording_number'].astype(int),
                  _map['SEX'].astype(str).str.upper().str[0]))


def anchors_for(QT_nomogram_value, RR_ms, sex):
    """The QT threshold, in ms, under each rule. Index 7 (dark orange) sits here."""
    thr = QTC_PROLONG_F if sex == 'F' else QTC_PROLONG_M
    RR_s = RR_ms / 1000.0
    return {
        'nom':  float(QT_nomogram_value),      # the nomogram curve, unchanged
        'qtcb': thr * np.sqrt(RR_s),           # QTc_B = QT / sqrt(RR)
        'qtcf': thr * RR_s ** (1.0 / 3.0),     # QTc_F = QT / RR^(1/3)
    }


# =============================================================================
#  main loop -- everything above the anchor block is unchanged from the original
# =============================================================================
for filename in os.listdir(input_folder):

    if not filename.lower().endswith('.csv'):
        continue

    csv_path = os.path.join(input_folder, filename)
    file_name = filename

    # ---- recording number and sex, from the filename ------------------------
    stem = ''.join(ch for ch in filename.split('.')[0] if ch.isdigit())
    if not stem:
        print(f'{filename}: no recording number in the filename, skipped')
        continue
    rec = int(stem)
    sex = SEX_OF.get(rec)
    if sex not in ('M', 'F'):
        print(f'{filename}: no usable SEX for recording {rec}, skipped')
        continue

    # ---- read + clean (unchanged) -------------------------------------------
    ECG = pd.read_csv(csv_path, skiprows=1)
    ECG.columns = ["time", "mv"]
    ECG_in_mv = ECG["mv"]
    ECG_signal = ECG["mv"]

    fs = 1000
    increment = 1000 / fs

    time = np.arange(0, len(ECG_signal), 1)
    time_s = time / fs

    filtered_ecg = remove_baseline_wander(ECG_signal, fs)
    cleaned_ecg = wavelet_denoise(filtered_ecg)
    cleaned_ecg = nk.ecg_clean(cleaned_ecg, sampling_rate=fs)

    lowpassed = scipy.signal.medfilt(cleaned_ecg, kernel_size=fs + 1)
    highpassed = cleaned_ecg - lowpassed

    R_peaks_index = refine_r_peaks(highpassed, detect_r_peaks(highpassed, fs), fs)

    highpassed_df = pd.DataFrame(highpassed)
    cleaned_ecg = highpassed_df[0]
    R_peaks_mv = cleaned_ecg[R_peaks_index]

    R_peaks_ms = R_peaks_index / fs * 1000
    RR_interval = [R_peaks_ms[i + 1] - R_peaks_ms[i] for i in range(len(R_peaks_ms) - 1)]
    RR = np.mean(RR_interval)
    HR = 60000 / RR

    _, rpeaks = nk.ecg_peaks(cleaned_ecg, sampling_rate=fs, method="neurokit")
    signal_dwt, waves_dwt = nk.ecg_delineate(cleaned_ecg, rpeaks, sampling_rate=fs,
                                             method="dwt", show=False, show_type="all")
    ECG_R_Peaks = rpeaks['ECG_R_Peaks']

    if np.isnan(HR):
        R_peaks_index = ECG_R_Peaks
        R_peaks_mv = cleaned_ecg[ECG_R_Peaks]
        R_peaks_ms = R_peaks_index / fs * 1000
        R_peaks_s = R_peaks_index / fs
        RR_interval = [R_peaks_ms[i + 1] - R_peaks_ms[i] for i in range(len(R_peaks_ms) - 1)]
    else:
        R_peaks_ms = R_peaks_index / fs * 1000
        R_peaks_s = R_peaks_index / fs

    RR = np.mean(RR_interval)
    HR = 60000 / RR

    # ---- the nomogram value (unchanged) -------------------------------------
    QTnomogram = pd.read_csv(
        "/content/drive/My Drive/Colab Notebooks/UKB_Research/XAI/QTnomogram.csv")
    HR_nomogram = QTnomogram['HR']
    QT_nomogram = QTnomogram['QT_pred']
    index_HR = np.argmin(np.abs(HR_nomogram - HR))
    QT_nomogram_value = QT_nomogram[index_HR]

    Q_wave_index = int(round(28 * fs / 1000))   # 28 ms at any sampling rate

    # ---- THREE anchors ------------------------------------------------------
    anchors = anchors_for(QT_nomogram_value, RR, sex)
    limits = {}
    for r, a in anchors.items():
        bands_above = BANDS_ABOVE_FAST if HR >= 85 else BANDS_ABOVE
        limits[r] = (a - BAND * BANDS_BELOW,          # LowerColourLimit
                     a + BAND * bands_above)          # UpperColourLimit / max
    ramp_to = max(hi for _, hi in limits.values())    # build the ramp once, long enough

    # ---- Rpeaks file, now carrying all three anchors ------------------------
    Rpeaks_df = pd.DataFrame({
        'R_peaks_index': R_peaks_index,
        'R_peaks_ms': R_peaks_ms,
        'R_peaks_s': R_peaks_s,
        'Rpeaks_mv': R_peaks_mv * 0.005,
        'RR': RR, 'HR': HR, 'SEX': sex,
        'QT_value': anchors['nom'],        # kept under the original name
        'anchor_nom': anchors['nom'],
        'anchor_qtcb': anchors['qtcb'],
        'anchor_qtcf': anchors['qtcf'],
        'Q_wave_index': Q_wave_index,
    })
    Rpeaks_df.to_csv(OUT_RPEAKS + file_name + "_.csv")

    # ---- the ms-since-Q ramp: ONE vector, rule-independent ------------------
    # colourVector[i] = milliseconds from that beat's Q onset. The rule only
    # decides which stretch of it survives clipping, so it is built once.
    length = len(ECG_in_mv)
    colourVector = np.full(length, np.nan)

    for R in range(len(R_peaks_index)):
        index = int(R_peaks_index[R]) - Q_wave_index
        n = int(ramp_to)
        lo = max(0, index)
        hi = min(length, index + n)
        if hi > lo:
            colourVector[lo:hi] = (np.arange(lo, hi) - index) * increment

    # ---- clip it three ways -------------------------------------------------
    vectors = {}
    for r in RULES:
        LowerColourlimit, UpperColourLimit = limits[r]
        v = colourVector.copy()
        v[(v < LowerColourlimit) | (v > UpperColourLimit)] = np.nan
        vectors[r] = v

    out = pd.DataFrame({
        'time': time_s,
        'mv': cleaned_ecg,
        'newcolourVector': vectors['nom'],        # alias: existing R code still works
        'newcolourVector_nom': vectors['nom'],
        'newcolourVector_qtcb': vectors['qtcb'],
        'newcolourVector_qtcf': vectors['qtcf'],
    })
    out.to_csv(OUT_SIGNAL + file_name + "_.csv")

    print(f'{rec} {sex} HR {HR:.1f} | anchors nom {anchors["nom"]:.0f} '
          f'qtcb {anchors["qtcb"]:.0f} qtcf {anchors["qtcf"]:.0f} ms')

print("Finished!")

# **Viz**

In [ ]:
# activate R magic
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages('ggplot2')
install.packages('ggridges')

In [ ]:
%%R
library(ggplot2)
library(ggridges)

In [ ]:
%%R

ECG_With_Colour_dir = "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/QT_coloured_ECG_signal_v2"
ECG_With_Colour_dir_files = list.files(path=ECG_With_Colour_dir, pattern="*.csv", full.names=TRUE)
print(length(ECG_With_Colour_dir_files))
print(ECG_With_Colour_dir_files[1])

In [ ]:
%%R
length(ECG_With_Colour_dir_files)

In [ ]:
%%R
CELLS <- c("beat_colour","beat_grey","beat_line",
           "lead_colour","lead_grey","lead_line")
OUT_ROOT <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/images/study"

have <- lapply(setNames(file.path(OUT_ROOT, CELLS), CELLS),
               function(d) basename(list.files(d, "\\.png$")))

cat("files per cell:\n")
for (c in CELLS) cat(sprintf("  %-12s %5d\n", c, length(have[[c]])))

recs <- sub("^(\\d+).*$", "\\1", basename(ECG_With_Colour_dir_files))
png  <- paste0(recs, ".png")
done <- Reduce(`&`, lapply(CELLS, function(c) png %in% have[[c]]))

cat(sprintf("\ncomplete: %d of %d recordings (%.1f%%)\n",
            sum(done), length(done), 100*mean(done)))
cat(sprintf("first incomplete index f = %d  (%s)\n",
            which(!done)[1], basename(ECG_With_Colour_dir_files[which(!done)[1]])))
cat(sprintf("remaining: %d recordings\n", sum(!done)))

In [ ]:
%%R
for (c in c("beat_colour","beat_grey","beat_line"))
  unlink(file.path(OUT_ROOT, c, "2748.png"))

In [ ]:
%%R
OUT_ROOT <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/images/study"
for (c in c("beat_colour","beat_grey","beat_line"))
  unlink(file.path(OUT_ROOT, c, "2748.png"))

In [ ]:
%%R
START_F <- 1944

In [ ]:
%%R

# =============================================================================
#  Renders the 3x2 study: {colour area, grey area, line only} x {beat, lead}
#
#      images/study/beat_colour/   <rec>.png    448 x 448,  coloured area + line
#      images/study/beat_grey/     <rec>.png    448 x 448,  flat grey area + line
#      images/study/beat_line/     <rec>.png    448 x 448,  line only (standard)
#      images/study/lead_colour/   <rec>.png    4480 x 448, 10 s rhythm strip
#      images/study/lead_grey/     <rec>.png    4480 x 448
#      images/study/lead_line/     <rec>.png    4480 x 448
#
#  Same palette, same grid, same y limits and same ms-per-pixel in both
#  layouts. Filenames are normalised to the ECG recording number, so the
#  cells join to the mapping table and to each other directly.
#
#  What the three fills isolate:
#      colour vs grey  -> hue, with filled area held constant
#      grey   vs line  -> the filled area itself, independent of hue
#      colour vs line  -> full effect vs the standard representation
# =============================================================================

library(ggplot2)
library(ggridges)        # geom_ridgeline_gradient
library(RColorBrewer)
library(grid)            # unit()

# ---- configuration ----------------------------------------------------------
BASE      <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ"
RPEAK_DIR <- file.path(BASE, "QT_coloured_ECG_signal_v2", "Rpeaks")
OUT_ROOT  <- file.path(BASE, "images", "study")

FIXED_WINDOW <- 1.6      # s, single-beat viewport. RR range 0.618-1.528 s
PAD          <- 0.04     # s before the R peak, as in the original code

LEAD_SEC  <- 10.0        # s shown in the rhythm strip

Y_LIM     <- c(-0.2, 1.0)
SHOW_GRID <- TRUE
IMG_DPI   <- 200

# beat: square, like a single-object image
BEAT_IN   <- 2.24                 # 2.24 in at 200 dpi = 448 x 448
# lead: 10:1 rhythm strip, as in the standard clinical lead II presentation
LEAD_IN_W <- 22.4                 # -> 4480 x 448
LEAD_IN_H <- 2.24

STYLES  <- c("colour", "grey", "line")
LAYOUTS <- c("beat", "lead")
CELLS   <- as.vector(outer(LAYOUTS, STYLES, paste, sep = "_"))

OUT_DIR <- setNames(file.path(OUT_ROOT, CELLS), CELLS)
for (d in OUT_DIR) dir.create(d, recursive = TRUE, showWarnings = FALSE)

# ---- palette ----------------------------------------------------------------
myColor <- rev(RColorBrewer::brewer.pal(11, "Spectral"))

make_fill_scale <- function(anchor) {
  colorcodesValues <- NA
  length(colorcodesValues) <- 9
  colorcodesValues[1] <- anchor - (40*6)  # purple (LowerColourLimit)
  colorcodesValues[2] <- anchor - (40*5)  # blue
  colorcodesValues[3] <- anchor - (40*4)  # lime
  colorcodesValues[4] <- anchor - (40*3)  # green
  colorcodesValues[5] <- anchor - (40*2)  # yellow
  colorcodesValues[6] <- anchor - (40*1)  # orange
  colorcodesValues[7] <- anchor           # dark orange = the QT nomogram value
  colorcodesValues[8] <- anchor + (40*1)  # red
  colorcodesValues[9] <- anchor + (40*2)  # dark red (UpperColourLimit)

  scale_fill_gradientn(
    colours = myColor,
    breaks  = c(colorcodesValues[1]+1, colorcodesValues[2:8], colorcodesValues[9]-1),
    labels  = as.character(colorcodesValues),
    limits  = c(colorcodesValues[1]+1, colorcodesValues[9]-1)
  )
}

# ---- shared layers ----------------------------------------------------------
fill_layers <- function(p, style, xvar, anchor) {
  switch(style,
    colour = p + geom_ridgeline_gradient(
                   aes(x = .data[[xvar]], y = 0, height = mv,
                       fill = newcolourVector),
                   gradient_lwd = 1) +
                 make_fill_scale(anchor),
    grey   = p + geom_ribbon(aes(ymin = 0, ymax = mv), fill = "grey60"),
    line   = p,                      # no fill: the line graph alone
    stop("unknown style: ", style)
  )
}

blank_theme <- function() {
  th <- theme(axis.text.x = element_blank(), axis.text.y = element_blank(),
              axis.line   = element_blank(), axis.ticks  = element_blank(),
              strip.text  = element_blank(), strip.background = element_blank(),
              panel.spacing = unit(0, "pt"),
              legend.position = "none")
  if (!SHOW_GRID) th <- th + theme(panel.grid = element_blank())
  th
}

save_png <- function(p, outfile, w, h) {
  ggsave(outfile, plot = p, width = w, height = h, dpi = IMG_DPI, bg = "white")
}

# ---- renderer: single beat --------------------------------------------------
render_beat <- function(dat, outfile, style, t0, rr, anchor) {

  lo <- t0 - PAD
  hi <- t0 + rr + PAD
  d  <- dat[dat$time >= lo & dat$time <= hi, ]     # one beat only
  if (nrow(d) == 0) return(FALSE)

  p <- ggplot(d, aes(x = time, y = mv))
  p <- fill_layers(p, style, "time", anchor)

  p <- p +
    geom_line(linewidth = 0.7) +
    scale_y_continuous(limits = Y_LIM, expand = c(0, 0),
                       minor_breaks = seq(-0.2, 1.0, 0.1),
                       breaks       = seq(-0.2, 1.0, 0.5)) +
    scale_x_continuous(minor_breaks = seq(0, 20, 0.04),
                       breaks       = seq(0, 20, 0.2),
                       expand       = c(0, 0)) +
    # viewport is a constant 1.6 s though only one beat is drawn:
    # the blank space to the right IS the heart rate
    coord_cartesian(xlim = c(lo, lo + FIXED_WINDOW)) +
    blank_theme() +
    labs(x = "", y = "", fill = "")

  save_png(p, outfile, BEAT_IN, BEAT_IN)
  TRUE
}

# ---- renderer: lead II rhythm strip -----------------------------------------
# 10 s on one row, 10:1 aspect, as lead II is presented clinically.
# The x window is a fixed 10 s for every recording, so the time scale is
# identical across images (unlike signal_time_seconds, which varied).
render_lead <- function(dat, outfile, style, anchor, t_start = 0) {

  d <- dat[dat$time >= t_start & dat$time < t_start + LEAD_SEC, ]
  if (nrow(d) == 0) return(FALSE)

  p <- ggplot(d, aes(x = time, y = mv))
  p <- fill_layers(p, style, "time", anchor)

  p <- p +
    geom_line(linewidth = 0.7) +
    scale_y_continuous(limits = Y_LIM, expand = c(0, 0),
                       minor_breaks = seq(-0.2, 1.0, 0.1),
                       breaks       = seq(-0.2, 1.0, 0.5)) +
    scale_x_continuous(minor_breaks = seq(0, 30, 0.04),
                       breaks       = seq(0, 30, 0.2),
                       expand       = c(0, 0)) +
    coord_cartesian(xlim = c(t_start, t_start + LEAD_SEC)) +
    blank_theme() +
    labs(x = "", y = "", fill = "")

  save_png(p, outfile, LEAD_IN_W, LEAD_IN_H)
  TRUE
}

# ---- main loop --------------------------------------------------------------
log_rows <- list()
n_files  <- length(ECG_With_Colour_dir_files)
t_start_run <- Sys.time()

for (f in START_F:n_files) {

  csv_path <- ECG_With_Colour_dir_files[f]
  ECG_name <- basename(csv_path)

  # normalise '900.csv_.csv' -> '900' so all six cells share one filename
  rec <- sub("^(\\d+).*$", "\\1", ECG_name)
  if (!grepl("^[0-9]+$", rec)) {
    message(sprintf("[%d/%d] %s: no leading recording number, skipped",
                    f, n_files, ECG_name))
    next
  }
  png_name <- paste0(rec, ".png")

  if (all(file.exists(file.path(unlist(OUT_DIR), png_name)))) next

  dat <- read.csv(csv_path)

  rpeak_path <- file.path(RPEAK_DIR, ECG_name)
  if (!file.exists(rpeak_path)) {
    message(sprintf("[%d/%d] %s: no Rpeaks file, skipped", f, n_files, ECG_name))
    next
  }
  Rpeaks    <- read.csv(rpeak_path)
  anchor    <- Rpeaks$QT_value[1]        # nomogram threshold, from mean HR
  R_peaks_s <- Rpeaks$R_peaks_s

  if (length(R_peaks_s) < 2) {
    message(sprintf("[%d/%d] %s: fewer than 2 R peaks, skipped", f, n_files, ECG_name))
    next
  }

  t0  <- R_peaks_s[1]
  rr  <- R_peaks_s[2] - R_peaks_s[1]
  dur <- max(dat$time, na.rm = TRUE)

  if (rr + 2 * PAD > FIXED_WINDOW)
    message(sprintf("  !! %s: RR %.3f s exceeds the %.2f s window",
                    ECG_name, rr, FIXED_WINDOW))
  if (dur < LEAD_SEC)
    message(sprintf("  !! %s: only %.2f s of signal, strip partly blank",
                    ECG_name, dur))

  ok <- TRUE
  for (layout in LAYOUTS) for (style in STYLES) {
    cell    <- paste(layout, style, sep = "_")
    outfile <- file.path(OUT_DIR[[cell]], png_name)
    if (file.exists(outfile)) next
    ok <- if (layout == "beat")
            render_beat(dat, outfile, style, t0, rr, anchor) && ok
          else
            render_lead(dat, outfile, style, anchor) && ok
  }

  log_rows[[length(log_rows) + 1]] <- data.frame(
    ECG_recording_number = as.integer(rec), source = ECG_name,
    R_peak_s = t0, RR_s = rr, duration_s = dur,
    n_Rpeaks = length(R_peaks_s), QT_nomogram = anchor, rendered = ok,
    stringsAsFactors = FALSE
  )

  if (f %% 50 == 0 || f == n_files) {
    el <- as.numeric(difftime(Sys.time(), t_start_run, units = "mins"))
    message(sprintf("[%d/%d] %.1f min elapsed, ~%.0f min remaining",
                    f, n_files, el, el / f * (n_files - f)))
  }
}

# ---- log --------------------------------------------------------------------
if (length(log_rows)) {
  log_df <- do.call(rbind, log_rows)
  write.csv(log_df, file.path(OUT_ROOT, "render_log.csv"), row.names = FALSE)
  message(sprintf("\n%d recordings rendered this run", nrow(log_df)))
  message(sprintf("RR (beat 1)  : %.3f - %.3f s", min(log_df$RR_s), max(log_df$RR_s)))
  message(sprintf("duration     : %.2f - %.2f s",
                  min(log_df$duration_s), max(log_df$duration_s)))
}

message("\nfiles per cell:")
for (cell in CELLS)
  message(sprintf("  %-16s %d", cell,
                  length(list.files(OUT_DIR[[cell]], "\\.png$"))))

In [ ]:
%%R
for (f in 1:length(ECG_With_Colour_dir_files))
{
    ECG_with_colourScale <- read.csv(ECG_With_Colour_dir_files[f])
    mv <- ECG_with_colourScale$mv
    time <- ECG_with_colourScale$time
    ECG_colour <- basename(ECG_With_Colour_dir_files[f])
    Rpeaks_file <- paste0("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/QT_coloured_ECG_signal_v2/Rpeaks/",ECG_colour)
    Rpeaks <- read.csv(Rpeaks_file)
    QT_nomogram_value <- Rpeaks$QT_value[1]
    R_peaks_s <- Rpeaks$R_peaks_s
    print(QT_nomogram_value)
    print(R_peaks_s)

    index = length(mv)
    signal_time_seconds = round(time[index-1])


    #next script is for visulising the ECG with the colour scale
    colorcodesValues <- NA
    length(colorcodesValues) <- 9
    colorcodesValues[1] <- QT_nomogram_value - (40*6) # purple. This also equals to the `LowerColourlimit' value
    colorcodesValues[2] <- QT_nomogram_value - (40*5) #blue
    colorcodesValues[3] <- QT_nomogram_value - (40*4) #lime
    colorcodesValues[4] <- QT_nomogram_value - (40*3) #green
    colorcodesValues[5] <- QT_nomogram_value - (40*2) #yellow
    colorcodesValues[6] <- QT_nomogram_value - (40*1) #orange
    colorcodesValues[7] <- QT_nomogram_value #dark orange. This is the QT nomogram value
    colorcodesValues[8] <- QT_nomogram_value + (40*1) #red
    colorcodesValues[9] <- QT_nomogram_value + (40*2) #dark red. This also equals to the `UpperColourLimit' value


    #create the pesudo colour vector using spectral codes
    myColor <- rev(RColorBrewer::brewer.pal(11, "Spectral"))
    myColor_scale_fill <- scale_fill_gradientn(colours = myColor,breaks=c(colorcodesValues[1]+1,colorcodesValues[2],colorcodesValues[3],colorcodesValues[4],colorcodesValues[5],
    colorcodesValues[6],colorcodesValues[7],colorcodesValues[8],colorcodesValues[9]-1),labels = c(as.character(unique(colorcodesValues[1])),colorcodesValues[2],colorcodesValues[3],colorcodesValues[4],colorcodesValues[5],colorcodesValues[6],colorcodesValues[7],colorcodesValues[8],as.character(unique(colorcodesValues[9]))),limits=c(colorcodesValues[1]+1,colorcodesValues[9]-1))

    # plot the ECG with the colour scale
    p <- ggplot(data=ECG_with_colourScale, aes(x=time, y=mv, fill=newcolourVector), binwidth=0)
    plot <- p  +
        scale_y_continuous(minor_breaks = seq(-0.2, +1.0, 0.1),breaks = seq(-0.2, +1.0, 0.1), lim = c(-0.2, +1.0)) +
        scale_x_continuous(minor_breaks = seq(0 , signal_time_seconds, 0.04),breaks = seq(0, signal_time_seconds, 0.2)) +

      #plot the coloured ECG + the ECG line graph (signal)
      #geom_bar(stat="identity", position ="dodge", colour=NA, size=0) +

      geom_ridgeline_gradient(aes(x=time, y = 0, height = mv, fill=newcolourVector), gradient_lwd = 1) +

      #facet_grid(~newcolourVector, scales = 'free_x', space = 'free') +
      geom_line(size=0.7) +
      myColor_scale_fill +
      theme(axis.text.x= element_blank()) +
      theme(axis.text.y= element_blank()) +
      theme(axis.line = element_blank(),axis.ticks = element_blank()) +
      #plot ECG baseline
      #geom_line(data=file2,aes(x=time, y=0), colour="#444444", lwd=0.5) +

      labs(x ="", y="", fill = "") + theme(legend.text=element_blank(),legend.title=element_blank(),legend.position = "none")
      Image <- paste0("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/images/lead/colour/",ECG_colour,"_.png")
      ggsave(Image, width=22.4, height=2.24, dpi = 300)

        for (i in 1:(length(R_peaks_s)-1))
        {
          p  +
          #plot vertical solid line for the R-peaks and dahsed line for the QT nomogram value
          scale_y_continuous(minor_breaks = seq(-0.2, +1.0, 0.1),breaks = seq(-0.2, +1.0, 0.5), lim = c(-0.2, +1.0)) +
          scale_x_continuous(minor_breaks = seq(0, 10, 0.04),breaks = seq(0, 10, 0.2), lim = c(R_peaks_s[i]-0.04,R_peaks_s[i+1]+0.04)) +

          #plot the coloured ECG + the ECG line graph (signal)
          #geom_bar(stat="identity", position ="dodge", colour=NA, size=0) +

          geom_ridgeline_gradient(aes(x=time, y = 0, height = mv, fill=newcolourVector), gradient_lwd = 1) +

          #facet_grid(~newcolourVector, scales = 'free_x', space = 'free') +
          geom_line(size=0.7) +
          myColor_scale_fill +
          theme(axis.text.x= element_blank()) +
          theme(axis.text.y= element_blank()) +
          theme(axis.line = element_blank(),axis.ticks = element_blank()) +

          #plot ECG baseline
          #geom_line(data=file2,aes(x=time, y=0), colour="#444444", lwd=0.5) +

          labs(x ="", y="", fill = "") + theme(legend.text=element_blank(),legend.title=element_blank(),legend.position = "none")

          newpathforimage <- paste0("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/images/single_heartbeat/colour/",ECG_colour)

          ggsave(paste0(newpathforimage,"_Heartbeat_", i,".png"), width=2.24, height=2.24, dpi = 300)

       }

}

In [ ]:
# =============================================================================
#  SECOND PASS: the QTcB and QTcF colour arms.
#
#  Run this after render_study.R has finished. It renders only the four new
#  cells and touches nothing that already exists:
#
#      images/study/beat_colour_qtcb/   <rec>.png    448 x 448
#      images/study/beat_colour_qtcf/   <rec>.png    448 x 448
#      images/study/lead_colour_qtcb/   <rec>.png    4480 x 448
#      images/study/lead_colour_qtcf/   <rec>.png    4480 x 448
#
#  Identical palette, geometry, grid, limits and line widths to the first
#  pass. The ONLY thing that differs is where index 7 (dark orange) is
#  anchored:
#
#      colour      (pass 1)  anchor = the QT nomogram threshold
#      colour_qtcb (here)    anchor = thr * sqrt(RR)        Bazett
#      colour_qtcf (here)    anchor = thr * RR^(1/3)        Fridericia
#
#  with thr = 450 ms for men, 460 ms for women -- the prolongation threshold.
#  QTc 500 then lands at colour index ~7.8-8.5, i.e. red, and the scale tops
#  out at index 9 without clipping.
#
#  RATE SOURCE: mean RR is computed from this pipeline's own R peaks, NOT
#  from the mapping table's RR column. The nomogram anchor in pass 1 came
#  from the same place, so all three colour arms share one rate source and
#  differ only in the correction rule. Sex is the only field taken from the
#  mapping table.
# =============================================================================

library(ggplot2)
library(ggridges)
library(RColorBrewer)
library(grid)

# ---- configuration ----------------------------------------------------------
BASE      <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ"
RPEAK_DIR <- file.path(BASE, "QT_coloured_ECG_signal_v2", "Rpeaks")
OUT_ROOT  <- file.path(BASE, "images", "study")
MAP_CSV   <- file.path(BASE, "mapping_table_with_outcome_and_distance.csv")

QTC_PROLONG_M <- 450     # ms, men
QTC_PROLONG_F <- 460     # ms, women

FIXED_WINDOW <- 1.6      # s, single-beat viewport
PAD          <- 0.04     # s before the R peak
LEAD_SEC     <- 10.0     # s shown in the rhythm strip

Y_LIM     <- c(-0.2, 1.0)
SHOW_GRID <- TRUE
IMG_DPI   <- 200
BEAT_IN   <- 2.24                 # 448 x 448
LEAD_IN_W <- 22.4                 # 4480 x 448
LEAD_IN_H <- 2.24

STYLES  <- c("colour_qtcb", "colour_qtcf")
LAYOUTS <- c("beat", "lead")
CELLS   <- as.vector(outer(LAYOUTS, STYLES, paste, sep = "_"))

OUT_DIR <- setNames(file.path(OUT_ROOT, CELLS), CELLS)
for (d in OUT_DIR) dir.create(d, recursive = TRUE, showWarnings = FALSE)

# ---- sex lookup -------------------------------------------------------------
MAP <- read.csv(MAP_CSV, stringsAsFactors = FALSE)
MAP <- MAP[!duplicated(MAP$ECG_recording_number),
           c("ECG_recording_number", "SEX", "RR")]
SEX_OF <- setNames(toupper(substr(as.character(MAP$SEX), 1, 1)),
                   as.character(MAP$ECG_recording_number))
RR_MAP <- setNames(suppressWarnings(as.numeric(MAP$RR)),
                   as.character(MAP$ECG_recording_number))   # for QC only
message(sprintf("mapping table: %d recordings, sex %s",
                nrow(MAP), paste(names(table(SEX_OF)), table(SEX_OF),
                                 sep = "=", collapse = " ")))

# ---- anchor: where index 7 (dark orange) sits -------------------------------
anchor_for <- function(style, rr_s, sex) {
  thr <- if (identical(sex, "F")) QTC_PROLONG_F else QTC_PROLONG_M
  switch(style,
    colour_qtcb = thr * sqrt(rr_s),      # QTc_B = QT / sqrt(RR)
    colour_qtcf = thr * rr_s^(1/3),      # QTc_F = QT / RR^(1/3)
    stop("unknown style: ", style))
}

# ---- palette (identical to pass 1) ------------------------------------------
myColor <- rev(RColorBrewer::brewer.pal(11, "Spectral"))

make_fill_scale <- function(anchor) {
  colorcodesValues <- NA
  length(colorcodesValues) <- 9
  colorcodesValues[1] <- anchor - (40*6)  # purple (LowerColourLimit)
  colorcodesValues[2] <- anchor - (40*5)  # blue
  colorcodesValues[3] <- anchor - (40*4)  # lime
  colorcodesValues[4] <- anchor - (40*3)  # green
  colorcodesValues[5] <- anchor - (40*2)  # yellow
  colorcodesValues[6] <- anchor - (40*1)  # orange
  colorcodesValues[7] <- anchor           # dark orange = prolongation threshold
  colorcodesValues[8] <- anchor + (40*1)  # red       ~ QTc 500
  colorcodesValues[9] <- anchor + (40*2)  # dark red  (UpperColourLimit)

  scale_fill_gradientn(
    colours = myColor,
    breaks  = c(colorcodesValues[1]+1, colorcodesValues[2:8], colorcodesValues[9]-1),
    labels  = as.character(colorcodesValues),
    limits  = c(colorcodesValues[1]+1, colorcodesValues[9]-1)
  )
}

blank_theme <- function() {
  th <- theme(axis.text.x = element_blank(), axis.text.y = element_blank(),
              axis.line   = element_blank(), axis.ticks  = element_blank(),
              strip.text  = element_blank(), strip.background = element_blank(),
              panel.spacing = unit(0, "pt"),
              legend.position = "none")
  if (!SHOW_GRID) th <- th + theme(panel.grid = element_blank())
  th
}

save_png <- function(p, outfile, w, h) {
  ggsave(outfile, plot = p, width = w, height = h, dpi = IMG_DPI, bg = "white")
}

# ---- renderers (identical geometry to pass 1) -------------------------------
render_beat <- function(dat, outfile, t0, rr, anchor) {
  lo <- t0 - PAD
  hi <- t0 + rr + PAD
  d  <- dat[dat$time >= lo & dat$time <= hi, ]
  if (nrow(d) == 0) return(FALSE)

  p <- ggplot(d, aes(x = time, y = mv)) +
    geom_ridgeline_gradient(
      aes(x = time, y = 0, height = mv, fill = newcolourVector),
      gradient_lwd = 1) +
    make_fill_scale(anchor) +
    geom_line(linewidth = 0.7) +
    scale_y_continuous(limits = Y_LIM, expand = c(0, 0),
                       minor_breaks = seq(-0.2, 1.0, 0.1),
                       breaks       = seq(-0.2, 1.0, 0.5)) +
    scale_x_continuous(minor_breaks = seq(0, 20, 0.04),
                       breaks       = seq(0, 20, 0.2),
                       expand       = c(0, 0)) +
    coord_cartesian(xlim = c(lo, lo + FIXED_WINDOW)) +
    blank_theme() +
    labs(x = "", y = "", fill = "")

  save_png(p, outfile, BEAT_IN, BEAT_IN)
  TRUE
}

render_lead <- function(dat, outfile, anchor, t_start = 0) {
  d <- dat[dat$time >= t_start & dat$time < t_start + LEAD_SEC, ]
  if (nrow(d) == 0) return(FALSE)

  p <- ggplot(d, aes(x = time, y = mv)) +
    geom_ridgeline_gradient(
      aes(x = time, y = 0, height = mv, fill = newcolourVector),
      gradient_lwd = 1) +
    make_fill_scale(anchor) +
    geom_line(linewidth = 0.7) +
    scale_y_continuous(limits = Y_LIM, expand = c(0, 0),
                       minor_breaks = seq(-0.2, 1.0, 0.1),
                       breaks       = seq(-0.2, 1.0, 0.5)) +
    scale_x_continuous(minor_breaks = seq(0, 30, 0.04),
                       breaks       = seq(0, 30, 0.2),
                       expand       = c(0, 0)) +
    coord_cartesian(xlim = c(t_start, t_start + LEAD_SEC)) +
    blank_theme() +
    labs(x = "", y = "", fill = "")

  save_png(p, outfile, LEAD_IN_W, LEAD_IN_H)
  TRUE
}

# ---- skip cache: 4 directory listings instead of ~21,000 file.exists calls ---
have <- lapply(OUT_DIR, function(d) basename(list.files(d, "\\.png$")))
names(have) <- CELLS
message(sprintf("already present: %s",
                paste(CELLS, sapply(have, length), sep = "=", collapse = " ")))

# ---- main loop --------------------------------------------------------------
log_rows <- list()
n_files  <- length(ECG_With_Colour_dir_files)
t_run    <- Sys.time()
n_nosex  <- 0

for (f in seq_len(n_files)) {

  csv_path <- ECG_With_Colour_dir_files[f]
  ECG_name <- basename(csv_path)

  rec <- sub("^(\\d+).*$", "\\1", ECG_name)
  if (!grepl("^[0-9]+$", rec)) next
  png_name <- paste0(rec, ".png")

  todo <- CELLS[!vapply(CELLS, function(c) png_name %in% have[[c]], logical(1))]
  if (length(todo) == 0) next

  sex <- SEX_OF[[rec]]
  if (is.null(sex) || is.na(sex) || !(sex %in% c("M", "F"))) {
    n_nosex <- n_nosex + 1
    message(sprintf("  !! %s: no usable SEX in the mapping table, skipped", ECG_name))
    next
  }

  rpeak_path <- file.path(RPEAK_DIR, ECG_name)
  if (!file.exists(rpeak_path)) next
  Rpeaks    <- read.csv(rpeak_path)
  R_peaks_s <- Rpeaks$R_peaks_s
  if (length(R_peaks_s) < 2) next

  # rate from THIS pipeline's own R peaks, matching the pass-1 nomogram anchor
  rr_all <- diff(R_peaks_s)
  rr_s   <- mean(rr_all, na.rm = TRUE)
  t0     <- R_peaks_s[1]
  rr1    <- rr_all[1]

  dat <- read.csv(csv_path)

  if (rr1 + 2 * PAD > FIXED_WINDOW)
    message(sprintf("  !! %s: beat-1 RR %.3f s exceeds the %.2f s window",
                    ECG_name, rr1, FIXED_WINDOW))

  ok <- TRUE
  for (cell in todo) {
    style   <- sub("^(beat|lead)_", "", cell)
    anchor  <- anchor_for(style, rr_s, sex)
    outfile <- file.path(OUT_DIR[[cell]], png_name)
    ok <- if (startsWith(cell, "beat"))
            render_beat(dat, outfile, t0, rr1, anchor) && ok
          else
            render_lead(dat, outfile, anchor) && ok
  }

  log_rows[[length(log_rows) + 1]] <- data.frame(
    ECG_recording_number = as.integer(rec), SEX = sex,
    RR_mean_s = rr_s, RR_beat1_s = rr1,
    RR_mapping_ms = unname(RR_MAP[[rec]]),          # for the QC comparison
    anchor_qtcb = anchor_for("colour_qtcb", rr_s, sex),
    anchor_qtcf = anchor_for("colour_qtcf", rr_s, sex),
    n_cells = length(todo), rendered = ok,
    stringsAsFactors = FALSE)

  if (f %% 50 == 0 || f == n_files) {
    el <- as.numeric(difftime(Sys.time(), t_run, units = "mins"))
    message(sprintf("[%d/%d] %.1f min elapsed, ~%.0f min remaining",
                    f, n_files, el, el / f * (n_files - f)))
  }
}

# ---- log + QC ---------------------------------------------------------------
if (length(log_rows)) {
  log_df <- do.call(rbind, log_rows)
  write.csv(log_df, file.path(OUT_ROOT, "render_log_qtc.csv"), row.names = FALSE)

  message(sprintf("\n%d recordings rendered this pass (%d skipped: no sex)",
                  nrow(log_df), n_nosex))
  message(sprintf("mean RR  : %.3f - %.3f s", min(log_df$RR_mean_s), max(log_df$RR_mean_s)))
  message(sprintf("anchor B : %.0f - %.0f ms", min(log_df$anchor_qtcb), max(log_df$anchor_qtcb)))
  message(sprintf("anchor F : %.0f - %.0f ms", min(log_df$anchor_qtcf), max(log_df$anchor_qtcf)))

  # QC: this pipeline's rate vs the mapping table's. Should agree closely.
  d <- log_df$RR_mean_s * 1000 - log_df$RR_mapping_ms
  d <- d[is.finite(d)]
  if (length(d))
    message(sprintf("RR vs mapping table: bias %+.1f ms, 95%% limits %+.1f to %+.1f ms",
                    mean(d), mean(d) - 1.96 * sd(d), mean(d) + 1.96 * sd(d)))
}

message("\nfiles per cell:")
for (cell in CELLS)
  message(sprintf("  %-22s %d", cell, length(list.files(OUT_DIR[[cell]], "\\.png$"))))

In [ ]:
import glob

os.chdir("/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/images/study/lead_colour/")
os.getcwd()

path=os.getcwd()
images=glob.glob(os.path.join(path,"*.png"))
len(images)

In [ ]:
%%R
CELLS <- c("beat_colour","beat_grey","beat_line",
           "lead_colour","lead_grey","lead_line")
OUT_ROOT <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ/images/study"

have <- lapply(setNames(file.path(OUT_ROOT, CELLS), CELLS),
               function(d) basename(list.files(d, "\\.png$")))

ref <- have[[1]]
cat("reference cell has", length(ref), "files\n\n")
for (c in CELLS) {
  miss  <- setdiff(ref, have[[c]])
  extra <- setdiff(have[[c]], ref)
  f     <- list.files(file.path(OUT_ROOT, c), "\\.png$", full.names = TRUE)
  cat(sprintf("  %-12s %5d | missing %d | extra %d | under 1KB %d\n",
              c, length(have[[c]]), length(miss), length(extra),
              sum(file.size(f) < 1000)))
}

# **Viz QTcB and QTcF**

In [ ]:
%%R
BASE      <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ"
RPEAK_DIR <- file.path(BASE, "QT_coloured_ECG_signal_v3", "Rpeaks")

ECG_With_Colour_dir_files <- list.files(
  file.path(BASE, "QT_coloured_ECG_signal_v3"),
  pattern = "\\.csv$", full.names = TRUE)

cat(length(ECG_With_Colour_dir_files), "signal CSVs found\n")
cat(length(list.files(RPEAK_DIR, "\\.csv$")), "Rpeaks files found\n")
print(head(basename(ECG_With_Colour_dir_files), 3))

In [ ]:
%%R
# =============================================================================
#  SECOND PASS: the QTcB and QTcF colour arms -- FOUR new cells only.
#
#  Run after the Python colouring has been regenerated with three vectors.
#  Touches nothing that already exists:
#
#      images/study/beat_colour_qtcb/   <rec>.png    448 x 448
#      images/study/beat_colour_qtcf/   <rec>.png    448 x 448
#      images/study/lead_colour_qtcb/   <rec>.png    4480 x 448
#      images/study/lead_colour_qtcf/   <rec>.png    4480 x 448
#
#  Identical palette, geometry, grid, limits and line widths to pass 1. Two
#  things differ per arm, and BOTH now come from Python so the two languages
#  cannot drift apart on the rate correction:
#
#      the colour column   newcolourVector_qtcb / _qtcf   (from the signal CSV)
#      the anchor          anchor_qtcb / anchor_qtcf      (from the Rpeaks CSV)
#
#  No sex lookup and no anchor arithmetic here -- Python already did both.
# =============================================================================

library(ggplot2)
library(ggridges)
library(RColorBrewer)
library(grid)

# ---- configuration ----------------------------------------------------------
BASE      <- "/content/drive/My Drive/Colab Notebooks/2026/ECGRDVQ"

RPEAK_DIR <- file.path(BASE, "QT_coloured_ECG_signal_v3", "Rpeaks")

ECG_With_Colour_dir_files <- list.files(
  file.path(BASE, "QT_coloured_ECG_signal_v3"), pattern = "\\.csv$", full.names = TRUE)

OUT_ROOT  <- file.path(BASE, "images", "study")

FIXED_WINDOW <- 1.6      # s, single-beat viewport
PAD          <- 0.04     # s before the R peak
LEAD_SEC     <- 10.0     # s shown in the rhythm strip

Y_LIM     <- c(-0.2, 1.0)
SHOW_GRID <- TRUE
IMG_DPI   <- 200
BEAT_IN   <- 2.24                 # 448 x 448
LEAD_IN_W <- 22.4                 # 4480 x 448
LEAD_IN_H <- 2.24

START_F <- 1                      # set higher to resume after a disconnect

RULES   <- c("qtcb", "qtcf")
LAYOUTS <- c("beat", "lead")
CELLS   <- as.vector(outer(LAYOUTS, paste0("colour_", RULES), paste, sep = "_"))

OUT_DIR <- setNames(file.path(OUT_ROOT, CELLS), CELLS)
for (d in OUT_DIR) dir.create(d, recursive = TRUE, showWarnings = FALSE)

# ---- palette (identical to pass 1) ------------------------------------------
myColor <- rev(RColorBrewer::brewer.pal(11, "Spectral"))

make_fill_scale <- function(anchor) {
  colorcodesValues <- NA
  length(colorcodesValues) <- 9
  colorcodesValues[1] <- anchor - (40*6)  # purple (LowerColourLimit)
  colorcodesValues[2] <- anchor - (40*5)  # blue
  colorcodesValues[3] <- anchor - (40*4)  # lime
  colorcodesValues[4] <- anchor - (40*3)  # green
  colorcodesValues[5] <- anchor - (40*2)  # yellow
  colorcodesValues[6] <- anchor - (40*1)  # orange
  colorcodesValues[7] <- anchor           # dark orange = prolongation threshold
  colorcodesValues[8] <- anchor + (40*1)  # red       ~ QTc 500
  colorcodesValues[9] <- anchor + (40*2)  # dark red  (UpperColourLimit)

  scale_fill_gradientn(
    colours = myColor,
    breaks  = c(colorcodesValues[1]+1, colorcodesValues[2:8], colorcodesValues[9]-1),
    labels  = as.character(colorcodesValues),
    limits  = c(colorcodesValues[1]+1, colorcodesValues[9]-1)
  )
}

blank_theme <- function() {
  th <- theme(axis.text.x = element_blank(), axis.text.y = element_blank(),
              axis.line   = element_blank(), axis.ticks  = element_blank(),
              strip.text  = element_blank(), strip.background = element_blank(),
              panel.spacing = unit(0, "pt"),
              legend.position = "none")
  if (!SHOW_GRID) th <- th + theme(panel.grid = element_blank())
  th
}

save_png <- function(p, outfile, w, h) {
  ggsave(outfile, plot = p, width = w, height = h, dpi = IMG_DPI, bg = "white")
}

# ---- renderers --------------------------------------------------------------
# `colvar` names the colour column for this rule: newcolourVector_qtcb / _qtcf
render_beat <- function(dat, outfile, colvar, t0, rr, anchor) {
  lo <- t0 - PAD
  hi <- t0 + rr + PAD
  d  <- dat[dat$time >= lo & dat$time <= hi, ]
  if (nrow(d) == 0) return(FALSE)

  p <- ggplot(d, aes(x = time, y = mv)) +
    geom_ridgeline_gradient(
      aes(x = time, y = 0, height = mv, fill = .data[[colvar]]),
      gradient_lwd = 1) +
    make_fill_scale(anchor) +
    geom_line(linewidth = 0.7) +
    scale_y_continuous(limits = Y_LIM, expand = c(0, 0),
                       minor_breaks = seq(-0.2, 1.0, 0.1),
                       breaks       = seq(-0.2, 1.0, 0.5)) +
    scale_x_continuous(minor_breaks = seq(0, 20, 0.04),
                       breaks       = seq(0, 20, 0.2),
                       expand       = c(0, 0)) +
    coord_cartesian(xlim = c(lo, lo + FIXED_WINDOW)) +
    blank_theme() +
    labs(x = "", y = "", fill = "")

  save_png(p, outfile, BEAT_IN, BEAT_IN)
  TRUE
}

render_lead <- function(dat, outfile, colvar, anchor, t_start = 0) {
  d <- dat[dat$time >= t_start & dat$time < t_start + LEAD_SEC, ]
  if (nrow(d) == 0) return(FALSE)

  p <- ggplot(d, aes(x = time, y = mv)) +
    geom_ridgeline_gradient(
      aes(x = time, y = 0, height = mv, fill = .data[[colvar]]),
      gradient_lwd = 1) +
    make_fill_scale(anchor) +
    geom_line(linewidth = 0.7) +
    scale_y_continuous(limits = Y_LIM, expand = c(0, 0),
                       minor_breaks = seq(-0.2, 1.0, 0.1),
                       breaks       = seq(-0.2, 1.0, 0.5)) +
    scale_x_continuous(minor_breaks = seq(0, 30, 0.04),
                       breaks       = seq(0, 30, 0.2),
                       expand       = c(0, 0)) +
    coord_cartesian(xlim = c(t_start, t_start + LEAD_SEC)) +
    blank_theme() +
    labs(x = "", y = "", fill = "")

  save_png(p, outfile, LEAD_IN_W, LEAD_IN_H)
  TRUE
}

# ---- skip cache: 4 directory listings, not ~21,000 file.exists calls --------
have <- lapply(OUT_DIR, function(d) basename(list.files(d, "\\.png$")))
names(have) <- CELLS
message(sprintf("already present: %s",
                paste(CELLS, sapply(have, length), sep = "=", collapse = " ")))

# ---- main loop --------------------------------------------------------------
log_rows <- list()
n_files  <- length(ECG_With_Colour_dir_files)
t_run    <- Sys.time()

for (f in START_F:n_files) {

  csv_path <- ECG_With_Colour_dir_files[f]
  ECG_name <- basename(csv_path)

  rec <- sub("^(\\d+).*$", "\\1", ECG_name)
  if (!grepl("^[0-9]+$", rec)) next
  png_name <- paste0(rec, ".png")

  todo <- CELLS[!vapply(CELLS, function(c) png_name %in% have[[c]], logical(1))]
  if (length(todo) == 0) next

  rpeak_path <- file.path(RPEAK_DIR, ECG_name)
  if (!file.exists(rpeak_path)) next
  Rpeaks <- read.csv(rpeak_path)

  if (!all(c("anchor_qtcb", "anchor_qtcf") %in% names(Rpeaks))) {
    message(sprintf("  !! %s: Rpeaks file has no anchor_qtcb/qtcf -- "
                    , ECG_name),
            "re-run the Python colouring first")
    next
  }

  R_peaks_s <- Rpeaks$R_peaks_s
  if (length(R_peaks_s) < 2) next
  t0  <- R_peaks_s[1]
  rr1 <- R_peaks_s[2] - R_peaks_s[1]

  dat <- read.csv(csv_path)

  if (rr1 + 2 * PAD > FIXED_WINDOW)
    message(sprintf("  !! %s: beat-1 RR %.3f s exceeds the %.2f s window",
                    ECG_name, rr1, FIXED_WINDOW))

  ok <- TRUE
  for (cell in todo) {
    rule    <- sub("^.*_colour_", "", cell)                 # "qtcb" / "qtcf"
    colvar  <- paste0("newcolourVector_", rule)             # the colour column
    anchor  <- Rpeaks[[paste0("anchor_", rule)]][1]         # from Python
    outfile <- file.path(OUT_DIR[[cell]], png_name)

    if (!colvar %in% names(dat)) {
      message(sprintf("  !! %s: signal CSV has no %s", ECG_name, colvar))
      ok <- FALSE; next
    }

    ok <- if (startsWith(cell, "beat"))
            render_beat(dat, outfile, colvar, t0, rr1, anchor) && ok
          else
            render_lead(dat, outfile, colvar, anchor) && ok
  }

  log_rows[[length(log_rows) + 1]] <- data.frame(
    ECG_recording_number = as.integer(rec),
    SEX = if ("SEX" %in% names(Rpeaks)) Rpeaks$SEX[1] else NA,
    RR_mean_ms = Rpeaks$RR[1], HR = Rpeaks$HR[1], RR_beat1_s = rr1,
    anchor_nom  = if ("anchor_nom" %in% names(Rpeaks)) Rpeaks$anchor_nom[1] else Rpeaks$QT_value[1],
    anchor_qtcb = Rpeaks$anchor_qtcb[1],
    anchor_qtcf = Rpeaks$anchor_qtcf[1],
    n_cells = length(todo), rendered = ok, stringsAsFactors = FALSE)

  if (f %% 50 == 0 || f == n_files) {
    el <- as.numeric(difftime(Sys.time(), t_run, units = "mins"))
    done <- f - START_F + 1
    message(sprintf("[%d/%d] %.1f min elapsed, ~%.0f min remaining",
                    f, n_files, el, el / done * (n_files - f)))
  }
}

# ---- log --------------------------------------------------------------------
if (length(log_rows)) {
  log_df <- do.call(rbind, log_rows)
  write.csv(log_df, file.path(OUT_ROOT, "render_log_qtc.csv"), row.names = FALSE)
  message(sprintf("\n%d recordings rendered this pass", nrow(log_df)))
  message(sprintf("anchor nom  : %.0f - %.0f ms", min(log_df$anchor_nom),  max(log_df$anchor_nom)))
  message(sprintf("anchor qtcb : %.0f - %.0f ms", min(log_df$anchor_qtcb), max(log_df$anchor_qtcb)))
  message(sprintf("anchor qtcf : %.0f - %.0f ms", min(log_df$anchor_qtcf), max(log_df$anchor_qtcf)))
  message(sprintf("median |nom - qtcb| = %.1f ms, |nom - qtcf| = %.1f ms",
                  median(abs(log_df$anchor_nom - log_df$anchor_qtcb)),
                  median(abs(log_df$anchor_nom - log_df$anchor_qtcf))))
}

message("\nfiles per cell:")
for (cell in CELLS)
  message(sprintf("  %-22s %d", cell, length(list.files(OUT_DIR[[cell]], "\\.png$"))))